<div dir="rtl" style="text-align:right">
<h1>InterviewPrep Agent – אבן דרך 1 (Milestone 1)</h1>
<p><b>מגישות:</b> דריה זוטובה ואלינה לוקיאנוב | <b>תחום:</b> Junior DevOps / Python</p>
<p>המחברת רק מריצה את הקוד שבמאגר (<code>src/</code>, <code>tests/</code>). היא רצה מתחילתה ועד סופה בקולאב נקי
(<b>Runtime → Run all</b>):</p>
<ol>
<li>שיבוט המאגר והתקנת <code>requirements.txt</code></li>
<li>טעינת מפתח Gemini מ-Colab Secrets – <b>רשות</b>. בלי מפתח הכול רץ עם המודל המדומה <code>MockLLM</code></li>
<li>הרצת כל הבדיקות בפקודה אחת (<code>pytest</code>)</li>
<li>הפקת קבצי העקבות (<code>docs/traces/</code>)</li>
<li>הדגמת לולאת הסוכן (מדומה, או Gemini כשמוגדר מפתח)</li>
<li>השוואת הקשר מלא מול הקשר מצומצם, ושתי שיחות שהמצב שלהן אינו מתערבב</li>
</ol>
</div>

<div dir="rtl" style="text-align:right">
<h2>שלב 1: שיבוט המאגר והתקנת התלויות</h2>
<p>התא משבט את המאגר <code>AlinaLu22/interview-prep-agent</code>. אם המחברת כבר רצה מתוך תיקיית המאגר, השיבוט מדולג.</p>
</div>

In [1]:
import os
import shutil
import subprocess
import sys
from pathlib import Path

REPO_URL = "https://github.com/AlinaLu22/interview-prep-agent.git"  # the submitted repository
REPO_DIR = "interview-prep-agent"

if not Path("src/agent.py").exists():  # clean Colab: the code is not here yet
    if Path(REPO_DIR, "src", "agent.py").exists():
        subprocess.run(["git", "-C", REPO_DIR, "pull", "--ff-only", "-q"])  # same session: take the latest version
    else:
        shutil.rmtree(REPO_DIR, ignore_errors=True)  # leftover of an earlier failed or empty clone
        subprocess.run(["git", "clone", "--depth", "1", REPO_URL, REPO_DIR], check=True)
    os.chdir(REPO_DIR)
assert Path("src/agent.py").exists(), "No src/ in the repository: upload the project files to GitHub first"
if os.getcwd() not in sys.path:
    sys.path.insert(0, os.getcwd())
print("Working directory:", os.getcwd())

Working directory: /content/interview-prep-agent


In [2]:
!pip install -q -r requirements.txt

<div dir="rtl" style="text-align:right">
<h2>שלב 2 (רשות): מודל אמיתי דרך Colab Secrets</h2>
<p>כדי להריץ עם Gemini: בסרגל הצד (אייקון המפתח) מוסיפים Secret בשם <code>GEMINI_API_KEY</code> ומפעילים
<b>Notebook access</b>. בלי מפתח התא מדפיס שהמחברת רצה עם המודל המדומה. המפתח לא נשמר במאגר.</p>
</div>

In [3]:
try:
    from google.colab import userdata
    _key = userdata.get("GEMINI_API_KEY")
    if _key:
        os.environ["GEMINI_API_KEY"] = _key
except Exception:  # not in Colab, secret missing, or notebook access not granted
    pass
# os.environ["GEMINI_MODEL"] = "gemini-3.5-flash"  # optional: force one model; by default the code
#                                                   # takes gemini-3.5-flash or the newest available Flash

from src.llm import make_llm
llm = make_llm()
print("Model for the demo:", f"Gemini ({llm.model})" if llm.name == "gemini" else "MockLLM (mock model, no API key)")

Model for the demo: MockLLM (mock model, no API key)


<div dir="rtl" style="text-align:right">
<h2>שלב 3: כל הבדיקות בפקודה אחת</h2>
<p>הבדיקות רצות תמיד עם מודל מדומה וכלים מדומים (מסומנים במפורש), בלי מפתח ובלי שירות חיצוני:
קלט תקין, קלט שגוי וכשל זמני לכל אחד מחמשת הכלים, מגבלת זמן, הרשאות, לולאת הסוכן, תנאי עצירה ובידוד בין שיחות.</p>
</div>

In [4]:
!python -m pytest -v

============================= test session starts ==============================
platform linux -- Python 3.13.18, pytest-8.3.3, pluggy-1.7.0 -- /usr/bin/python3
cachedir: .pytest_cache
rootdir: /content/interview-prep-agent
configfile: pytest.ini
testpaths: tests
plugins: platformdirs-4.14.3, anyio-4.16.0, typeguard-4.6.0, langsmith-0.14.17
collected 89 items                                                             

tests/test_agent.py::test_normal_run_matches_the_proposal_scenario PASSED [  1%]
tests/test_agent.py::test_approval_gate_blocks_save_without_user_approval PASSED [  2%]
tests/test_agent.py::test_model_cannot_approve_by_itself PASSED          [  3%]
tests/test_agent.py::test_step_budget_stops_the_loop PASSED              [  4%]
tests/test_agent.py::test_transient_failure_recovers_with_retry PASSED   [  5%]
tests/test_agent.py::test_tool_unavailable_stops_without_inventing_or_saving PASSED [  6%]
tests/test_agent.py::test_empty_resume_asks_for_clarification PASSED     [ 

<div dir="rtl" style="text-align:right">
<h2>שלב 4: הפקת קבצי העקבות</h2>
<p>שתי ריצות עם <code>MockLLM</code> על התרחיש הרגיל מההצעה: ריצה תקינה (10 צעדים מתוך תקציב של 14),
וריצה שבה הקריאה הראשונה ל-<code>search_question_bank</code> נכשלת בכשל זמני מדומה (503) ומצליחה בניסיון החוזר
(11 צעדים: גם הניסיון החוזר נספר).</p>
</div>

In [5]:
from src.agent import generate_submission_traces

generate_submission_traces("docs/traces")
for name in ("trace_normal.jsonl", "trace_transient.jsonl"):
    print(f"=== docs/traces/{name}")
    for line in Path("docs/traces", name).read_text(encoding="utf-8").splitlines():
        e = __import__("json").loads(line)
        if e["phase"] == "tool_call":
            print(f'step {e["step"]:>2} | {e["decided_by"]} -> {e["tool"]:<21} | {e["status"]:<5} | '
                  f'attempts={e["attempts"]} {e["retries"] or ""}')
        else:
            print(f'   {e["phase"]}: ' + ", ".join(f"{k}={v}" for k, v in e.items() if k != "phase"))

=== docs/traces/trace_normal.jsonl
   input: thread_id=thread-normal-01, user_id=daria, model=mock, max_steps=14
step  1 | mock -> parse_resume          | ok    | attempts=1 
step  2 | mock -> parse_vacancy         | ok    | attempts=1 
step  3 | mock -> compute_gap           | ok    | attempts=1 
step  4 | mock -> search_question_bank  | ok    | attempts=1 
step  5 | mock -> search_question_bank  | ok    | attempts=1 
step  6 | mock -> search_question_bank  | ok    | attempts=1 
step  7 | mock -> search_question_bank  | ok    | attempts=1 
step  8 | mock -> search_question_bank  | ok    | attempts=1 
   approval_gate: approved=True, question_count=10
step  9 | mock -> save_prep_plan        | ok    | attempts=1 
   final_decision: step=10, decided_by=mock
   done: status=DONE, stop_reason=model_final_answer, steps_used=10, max_steps=14, retries=0, question_count=10, saved=True
=== docs/traces/trace_transient.jsonl
   input: thread_id=thread-transient-01, user_id=alina, model=mock, max_

<div dir="rtl" style="text-align:right">
<h2>שלב 5: הדגמת לולאת הסוכן</h2>
<p>המודל (מדומה או Gemini) בוחר את הכלי הבא; הקוד מזין נתונים מהימנים מהמצב, מאמת, בודק הרשאות ומגבלת זמן,
ומבקש מהמשתמש אישור לפני השמירה. כאן האישור ניתן מראש (<code>approve_save=True</code>).</p>
<p>עם מפתח חינמי של Gemini (5 בקשות לדקה) השלב לוקח כ-3 דקות: הקוד ממתין בין הבקשות ומדפיס <code>[gemini] … waiting</code>.
בלי מפתח הוא מסתיים מיד.</p>
</div>

In [6]:
import textwrap

from src.agent import SAMPLE_RESUME, SAMPLE_VACANCY, run_interview_agent
from src.llm import MockLLM
from src.state import NoteStore, ThreadStore


def demo_run(model):
    return run_interview_agent("daria", f"colab-demo-{model.name}", SAMPLE_RESUME, SAMPLE_VACANCY,
                               ThreadStore(), NoteStore(), llm=model, approve_save=True)


try:
    demo = demo_run(llm)
except Exception as e:  # e.g. quota or model name: show the error and finish with the mock model
    print(textwrap.fill(f"Gemini run failed: {type(e).__name__}: {e}", 110)[:1500] + "\nFalling back to MockLLM.")
    demo = demo_run(MockLLM())

print("status:", demo["status"])
for e in demo["trace"]:
    if e["phase"] == "tool_call":
        print(f'step {e["step"]:>2} | {e["decided_by"]} -> {e["tool"]} {e["model_args"]} | {e["status"]}')
print("\n" + demo["answer"])
if demo["model_message"]:
    print("Model message:", demo["model_message"])
print("\nFirst 3 questions:")
for q in demo["questions"][:3]:
    print(f'- [{q["topic"]} | {"GAP" if q["is_gap"] else "MATCHED"}] {q["question"]}')

status: DONE
step  1 | mock -> parse_resume {} | ok
step  2 | mock -> parse_vacancy {} | ok
step  3 | mock -> compute_gap {} | ok
step  4 | mock -> search_question_bank {'skill': 'docker', 'top_k': 2} | ok
step  5 | mock -> search_question_bank {'skill': 'ci/cd', 'top_k': 2} | ok
step  6 | mock -> search_question_bank {'skill': 'python', 'top_k': 2} | ok
step  7 | mock -> search_question_bank {'skill': 'linux', 'top_k': 2} | ok
step  8 | mock -> search_question_bank {'skill': 'teamwork', 'category': 'behavioral', 'top_k': 2} | ok
step  9 | mock -> save_prep_plan {'plan_name': 'daria_colab-demo-mock_plan'} | ok

Target role: Junior DevOps Engineer
Matched skills: python, linux
Missing skills (gaps): docker, ci/cd
Weak skills: none
Prepared 10 grounded questions.
Saved approved plan to: prep_plans/daria_colab-demo-mock_plan.json
Model message: Preparation plan is ready.

First 3 questions:
- [Docker | GAP] What is the difference between a Docker image and a container? How do you keep ima

<div dir="rtl" style="text-align:right">
<h2>שלב 6: השוואת הקשר מלא מול הקשר מצומצם</h2>
<p>אותה שאלה (בסוף שיחה של 15 הודעות) נשלחת למודל פעם עם ההיסטוריה המלאה ופעם עם הקשר מצומצם.
הטבלה מציגה את גודל ההקשר, את תשובת המודל, האם היא נכונה, ומה אבד. עם מפתח – מוצגת גם טבלה של Gemini.</p>
</div>

In [7]:
import pandas as pd
from src.state import COMPARISON_QUESTION, compare_context_strategies

pd.set_option("display.max_colwidth", 90)
print("Question:", COMPARISON_QUESTION)
models = [MockLLM()] + ([llm] if llm.name == "gemini" else [])
for model in models:
    try:
        display(pd.DataFrame(compare_context_strategies(model)))
    except Exception as e:
        print(textwrap.fill(f"{model.name} comparison failed: {type(e).__name__}: {e}", 110)[:1500])

Question: Remind me which role am I preparing for, and which skills am I missing?


,strategy,model,messages_in_context,context_items,context_chars,approx_tokens,answer,correct,lost_in_context
0,full,mock,15,16,1097,272,"You are preparing for junior devops. Missing skills: docker, ci/cd.",True,[]
1,last_n (2),mock,2,3,319,80,I don't know: the current context does not say which role or skills.,False,"[role (junior devops), missing: docker, missing: ci/cd]"
2,notes + last_n (2),mock,2,5,390,98,"You are preparing for junior devops. Missing skills: docker, ci/cd.",True,[]


<div dir="rtl" style="text-align:right">
<h2>שלב 7: שתי שיחות שהמצב שלהן אינו מתערבב</h2>
<p>שני משתמשים, שתי שיחות, קורות חיים שונים. לכל שיחה מצב משלה (כולל יומן הקריאות לכלים),
ומשתמש אחר אינו יכול לפתוח שיחה שאינה שלו. בזיכרון בין שיחות נשמרות רק עובדות מתוכנית שאושרה,
והטקסט הגולמי של קורות החיים לא נשמר אחרי הריצה. אותה בדיקה רצה גם ב-<code>tests/test_agent.py</code>.</p>
</div>

In [8]:
threads, notes = ThreadStore(), NoteStore()
other_resume = "Mid DevOps engineer.\n4 years Docker and Kubernetes containers, CI/CD with Jenkins pipelines."
run_interview_agent("daria", "t-a", SAMPLE_RESUME, SAMPLE_VACANCY, threads, notes, approve_save=True)
run_interview_agent("alina", "t-b", other_resume, SAMPLE_VACANCY, threads, notes, approve_save=True)
for user, tid in (("daria", "t-a"), ("alina", "t-b")):
    s = threads.get_or_create(tid, user)
    print(f"{tid} ({user}): missing={s.gap_json['missing']} | steps={s.step} | raw resume kept={bool(s.resume_text)}"
          f" | notes={notes.all(user)}")
try:
    threads.get_or_create("t-a", "alina")
except PermissionError as e:
    print("alina -> t-a:", e)

t-a (daria): missing=['docker', 'ci/cd'] | steps=10 | raw resume kept=False | notes=['Target role: Junior DevOps Engineer', 'Missing skills: docker, ci/cd']
t-b (alina): missing=['python', 'linux'] | steps=10 | raw resume kept=False | notes=['Target role: Junior DevOps Engineer', 'Missing skills: python, linux']
alina -> t-a: thread belongs to another user
